In [2]:
# Ô 1 — Import
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# CountVectorizer: công cụ túi từ, chỉ đếm số lần xuất hiện.
# TfidfVectorizer: công cụ TF-IDF, đếm rồi đánh trọng số.
# cosine_similarity: đo độ giống nhau giữa hai hàng số, dùng ở Ô 12.

In [ ]:
# Ô 2 — Vì sao không đưa chữ thẳng vào mô hình được
from sklearn.linear_model import LogisticRegression

docs = ["free prize call now", "call me later", "free free entry win prize", "see you later"]
labels = [1, 0, 1, 0]   # 1 = spam, 0 = tin thường

try:
    # .reshape(-1, 1): biến list thành bảng 4 dòng × 1 cột, đúng hình dạng mà mô hình chờ.
    print(np.array(docs).reshape(-1, 1))
    LogisticRegression().fit(np.array(docs).reshape(-1, 1), labels) # Logistic Regression cần nhân hệ số với feature, mà không ai nhân được 0,5 × "free prize call now". Vì vậy nó báo lỗi không đổi được chữ thành số.
except ValueError as e:
    print("Lỗi:", str(e)[:60])
# Lỗi: could not convert string to float: 'free prize call now'

[['free prize call now']
 ['call me later']
 ['free free entry win prize']
 ['see you later']]
Lỗi: could not convert string to float: np.str_('free prize call 


In [ ]:
# Ô 3 — Bước 1: tách câu thành token
tach = CountVectorizer().build_analyzer()
print(tach("FREE Prize! Call now..."))

# Mặc định, bộ tách từ:
# Chuyển về chữ thường: FREE thành free, nên FREE và free được đếm chung
# Bỏ dấu câu: ! và ... biến mất.
# Bỏ từ chỉ có một ký tự, như a hay I.

['free', 'prize', 'call', 'now']


In [6]:
# Ô 4 — Bước 2: túi từ (bag of words)
cnt = CountVectorizer()
X_count = cnt.fit_transform(docs)
print(X_count)
# fit_transform(docs) làm hai việc một lúc : 
# fit: đọc cả 4 câu để học vocabulary, tức danh sách 10 từ khác nhau.
# transform: biến mỗi câu thành một hàng 10 số.


print(cnt.get_feature_names_out().tolist())
# get_feature_names_out(): tên các cột, sắp theo bảng chữ cái.
# ['call', 'entry', 'free', 'later', 'me', 'now', 'prize', 'see', 'win', 'you']
# X_count.toarray(): đổi sang bảng thường để in ra xem
pd.DataFrame(X_count.toarray(), columns=cnt.get_feature_names_out())
#    call  entry  free  later  me  now  prize  see  win  you
# 0     1      0     1      0   0    1      1    0    0    0   <- free prize call now
# 1     1      0     0      1   1    0      0    0    0    0   <- call me later
# 2     0      1     2      0   0    0      1    0    1    0   <- free free entry win prize
# 3     0      0     0      1   0    0      0    1    0    1   <- see you later

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 14 stored elements and shape (4, 10)>
  Coords	Values
  (0, 2)	1
  (0, 6)	1
  (0, 0)	1
  (0, 5)	1
  (1, 0)	1
  (1, 4)	1
  (1, 3)	1
  (2, 2)	2
  (2, 6)	1
  (2, 1)	1
  (2, 8)	1
  (3, 3)	1
  (3, 7)	1
  (3, 9)	1
['call', 'entry', 'free', 'later', 'me', 'now', 'prize', 'see', 'win', 'you']


,call,entry,free,later,me,now,prize,see,win,you
0,1,0,1,0,0,1,1,0,0,0
1,1,0,0,1,1,0,0,0,0,0
2,0,1,2,0,0,0,1,0,1,0
3,0,0,0,1,0,0,0,1,0,1


In [ ]:
# Ô 5 — Túi từ bỏ qua thứ tự và bỏ qua từ lạ
v = CountVectorizer().fit(["dog bites man"])
print(v.transform(["dog bites man", "man bites dog"]).toarray().tolist())
# [[1, 1, 1], [1, 1, 1]]   <- hai câu nghĩa ngược nhau, nhưng vector giống hệt

print(cnt.transform(["free pizza now"]).toarray().tolist())
# Từ lạ bị bỏ qua: transform chỉ dùng vocabulary đã học lúc fit. Chữ pizza không có trong 4 câu ban đầu nên không có cột cho nó và bị bỏ qua. 
# Đây gọi là từ ngoài vocabulary (out-of-vocabulary).
# [[0, 0, 1, 0, 0, 1, 0, 0, 0, 0]]   <- 'pizza' biến mất

[[1, 1, 1], [1, 1, 1]]
[[0, 0, 1, 0, 0, 1, 0, 0, 0, 0]]


In [ ]:
# Ô 6 — Bước 3: tính IDF bằng tay
doc_freq = (X_count.toarray() > 0).sum(axis=0)
print(dict(zip(cnt.get_feature_names_out(), doc_freq)))
# call 2, entry 1, free 2, later 2, me 1, now 1, prize 2, see 1, win 1, you 1

n = len(docs)
idf_tay = np.log((1 + n) / (1 + doc_freq)) + 1
print(dict(zip(cnt.get_feature_names_out(), idf_tay.round(3))))
# call 1.511, free 1.511, later 1.511, prize 1.511   (có ở 2 câu)
# entry 1.916, me 1.916, now 1.916, see 1.916, win 1.916, you 1.916   (chỉ ở 1 câu)

tfidf = TfidfVectorizer().fit(docs)
print(tfidf.idf_.round(3))   # cùng kết quả với cách tính tay

# IDF = ln( (1 + số câu) / (1 + số câu chứa từ) ) + 1

{'call': np.int64(2), 'entry': np.int64(1), 'free': np.int64(2), 'later': np.int64(2), 'me': np.int64(1), 'now': np.int64(1), 'prize': np.int64(2), 'see': np.int64(1), 'win': np.int64(1), 'you': np.int64(1)}
{'call': np.float64(1.511), 'entry': np.float64(1.916), 'free': np.float64(1.511), 'later': np.float64(1.511), 'me': np.float64(1.916), 'now': np.float64(1.916), 'prize': np.float64(1.511), 'see': np.float64(1.916), 'win': np.float64(1.916), 'you': np.float64(1.916)}
[1.511 1.916 1.511 1.511 1.916 1.916 1.511 1.916 1.916 1.916]


In [ ]:
# Ô 7 — Bước 4: tính TF-IDF của một câu bằng tay
tf = X_count.toarray()[0]          # TF: mỗi từ xuất hiện 1 lần
diem = tf * tfidf.idf_             # nhân TF với IDF
print(diem.round(3))
# call 1.511, free 1.511, now 1.916, prize 1.511, các cột khác 0

do_dai = np.sqrt((diem ** 2).sum())
print(round(do_dai, 3))            # 3.243
print((diem / do_dai).round(3))
# call 0.466, free 0.466, now 0.591, prize 0.466

# TF × IDF: mỗi từ có mặt trong câu được nhân với trọng số IDF của nó. now hiếm hơn (chỉ ở câu này) nên điểm cao hơn.
# Chuẩn hóa độ dài (L2 normalization): chia mọi số cho độ dài của vector, tính bằng căn bậc hai của tổng bình phương, ở đây là 3,243. Sau bước này, mỗi câu có “độ dài” bằng 1.
# Vì sao phải chuẩn hóa: không chuẩn hóa thì câu dài (nhiều từ) sẽ có điểm lớn hơn câu ngắn chỉ vì dài hơn. Chuẩn hóa đưa mọi câu về cùng thang để so sánh công bằng.

[1.511 0.    1.511 0.    0.    1.916 1.511 0.    0.    0.   ]
3.243
[0.466 0.    0.466 0.    0.    0.591 0.466 0.    0.    0.   ]


In [11]:
# Ô 8 — Bảng TF-IDF đầy đủ
X_tfidf = tfidf.transform(docs)
pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out()).round(2)
#    call  entry  free  later    me   now  prize   see   win   you
# 0  0.47   0.00  0.47   0.00  0.00  0.59   0.47  0.00  0.00  0.00
# 1  0.53   0.00  0.00   0.53  0.67  0.00   0.00  0.00  0.00  0.00
# 2  0.00   0.44  0.70   0.00  0.00  0.00   0.35  0.00  0.44  0.00
# 3  0.00   0.00  0.00   0.49  0.00  0.00   0.00  0.62  0.00  0.62

,call,entry,free,later,me,now,prize,see,win,you
0,0.47,0.00,0.47,0.00,0.00,0.59,0.47,0.00,0.00,0.00
1,0.53,0.00,0.00,0.53,0.67,0.00,0.00,0.00,0.00,0.00
2,0.00,0.44,0.70,0.00,0.00,0.00,0.35,0.00,0.44,0.00
3,0.00,0.00,0.00,0.49,0.00,0.00,0.00,0.62,0.00,0.62


In [12]:
# Ô 9 — Ma trận thưa (sparse matrix) trên dữ liệu thật
URL = ("https://raw.githubusercontent.com/justmarkham/"
       "pycon-2016-tutorial/master/data/sms.tsv")
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])

v_sms = TfidfVectorizer()
X_sms = v_sms.fit_transform(sms["message"])
print(type(X_sms).__name__, X_sms.shape)   # csr_matrix (5572, 8713)
print(X_sms.nnz)                            # 74169   ô khác 0
print(round(X_sms.nnz / (X_sms.shape[0] * X_sms.shape[1]) * 100, 3))   # 0.153 (%)

csr_matrix (5572, 8713)
74169
0.153


In [ ]:
# Ô 10 — IDF trên dữ liệu thật: từ nào nhẹ, từ nào nặng
idf = pd.Series(v_sms.idf_, index=v_sms.get_feature_names_out()).sort_values()
print(idf.head(5).round(2).to_dict())
# IDF thấp nhất là các từ có ở khắp nơi (to, you, the): gần như không giúp phân biệt tin nào với tin nào.
print("-----------")

for w in ["free", "claim", "prize", "lunch"]:
    print(w, round(idf[w], 2))

print("-----------")
# à các từ chỉ xuất hiện đúng một lần trong cả bộ: lỗi chính tả, ký tự lạ. Chúng hiếm nhưng thường là nhiễu. Ô 11 cho cách lọc bớt.
print(idf.tail(3).round(2).to_dict()) 


{'to': 2.19, 'you': 2.25, 'the': 2.68, 'in': 2.93, 'and': 2.95}
-----------
free 4.19
claim 4.93
prize 5.18
lunch 5.94
-----------
{'ú1': 8.93, '〨ud': 8.93, '000pes': 8.93}


In [19]:
# Ô 11 — Các tham số hay dùng của TfidfVectorizer
cau_hinh = {
    "mặc định": {},
    "stop_words='english'": {"stop_words": "english"},
    "min_df=2": {"min_df": 2},
    "ngram_range=(1, 2)": {"ngram_range": (1, 2)},
    "ngram (1,2) + min_df=2": {"ngram_range": (1, 2), "min_df": 2},
}
for ten, kw in cau_hinh.items():
    print(f"{ten:24s} -> {len(TfidfVectorizer(**kw).fit(sms['message']).vocabulary_)} từ")
# mặc định                 -> 8713 từ
# stop_words='english'     -> 8444 từ
# min_df=2                 -> 4213 từ
# ngram_range=(1, 2)       -> 50506 từ
# ngram (1,2) + min_df=2   -> 14993 từ

print(TfidfVectorizer(ngram_range=(1, 2)).build_analyzer()("call now to claim"))
# ['call', 'now', 'to', 'claim', 'call now', 'now to', 'to claim']

mặc định                 -> 8713 từ
stop_words='english'     -> 8444 từ
min_df=2                 -> 4213 từ
ngram_range=(1, 2)       -> 50506 từ
ngram (1,2) + min_df=2   -> 14993 từ
['call', 'now', 'to', 'claim', 'call now', 'now to', 'to claim']


In [20]:
# Ô 12 — Giới hạn lớn nhất: TF-IDF không hiểu nghĩa
cau = ["kill the process", "terminate the process", "open the file"]
v_nghia = TfidfVectorizer().fit(cau)
print(cosine_similarity(v_nghia.transform(cau)).round(2))
# [[1.   0.48 0.16]
#  [0.48 1.   0.16]
#  [0.16 0.16 1.  ]]

[[1.   0.48 0.16]
 [0.48 1.   0.16]
 [0.16 0.16 1.  ]]


In [21]:
# Ô 13 — Liên hệ M3: tokenizer mặc định không hợp với log
logs = [r"C:\Windows\System32\cmd.exe /c whoami",
        r"C:\Windows\System32\cmd.exe /c ipconfig",
        r"C:\Windows\System32\cmd.exe /c net user",
        r"C:\Windows\System32\sdbinst.exe -q payload.sdb"]

mac_dinh = TfidfVectorizer().fit(logs)
print(mac_dinh.get_feature_names_out().tolist())
# ['cmd', 'exe', 'ipconfig', 'net', 'payload', 'sdb', 'sdbinst', 'system32', 'user', 'whoami', 'windows']

cho_log = TfidfVectorizer(token_pattern=r"(?u)[\w\.\-]+").fit(logs)
print(cho_log.get_feature_names_out().tolist())
# ['-q', 'c', 'cmd.exe', 'ipconfig', 'net', 'payload.sdb', 'sdbinst.exe', 'system32', 'user', 'whoami', 'windows']

print(pd.Series(cho_log.idf_, index=cho_log.get_feature_names_out()).round(2).sort_values().to_dict())
# c 1.0, system32 1.0, windows 1.0 | cmd.exe 1.22 | sdbinst.exe 1.92, whoami 1.92, ...

['cmd', 'exe', 'ipconfig', 'net', 'payload', 'sdb', 'sdbinst', 'system32', 'user', 'whoami', 'windows']
['-q', 'c', 'cmd.exe', 'ipconfig', 'net', 'payload.sdb', 'sdbinst.exe', 'system32', 'user', 'whoami', 'windows']
{'c': 1.0, 'system32': 1.0, 'windows': 1.0, 'cmd.exe': 1.22, 'ipconfig': 1.92, 'net': 1.92, 'payload.sdb': 1.92, '-q': 1.92, 'sdbinst.exe': 1.92, 'user': 1.92, 'whoami': 1.92}
